## Import Packages

In [1]:
import pandas as pd

## Load Data

In [2]:
# Load historical data
HittersHistorical = pd.read_csv("../../HistoricalData/HittersHistorical.csv")
PithersHistorical = pd.read_csv("../../HistoricalData/PitchersHistorical.csv")

# Load ADP data with latin1 to safely read the special characters
HittersADP = pd.read_csv("AverageDraftPositionDataRAW/HittersADP2026RAW.csv", encoding='latin1')
PitchersADP = pd.read_csv("AverageDraftPositionDataRAW/PitchersADP2026RAW.csv", encoding='latin1')

# Display top elements of dataframe
print(HittersADP.head())
print()
print()
print(PitchersADP.head())

      Player Name Team     Position   ADP
0   Shohei Ohtani  LAD        SP,DH  1.17
1     Aaron Judge  NYY  LF,CF,RF,DH  1.83
2  Bobby Witt Jr.   KC           SS  4.00
3       Juan Soto  NYM        LF,RF  3.50
4    Jose Ramirez  CLE        3B,DH  5.50


          Player Name Team Position    ADP
0        Tarik Skubal  DET       SP   6.00
1         Paul Skenes  PIT       SP   8.50
2     Garrett Crochet  BOS       SP  11.33
3  Yoshinobu Yamamoto  LAD       SP  22.67
4  Cristopher Sanchez  PHI       SP  26.33


## Clean Data

#### Remove Accent Marks

In [3]:
# This automatically turns "Acuña" into "Acuna" and "José" into "Jose"
HittersADP['Player Name'] = HittersADP['Player Name'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
PitchersADP['Player Name'] = PitchersADP['Player Name'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')

print("Accent marks successfully removed!")

Accent marks successfully removed!


#### Normalize Team Abbreviations

In [4]:
TeamNames = {
    'ARI': 'Arizona Diamondbacks', 'ATL': 'Atlanta Braves', 'BAL': 'Baltimore Orioles',
    'BOS': 'Boston Red Sox', 'CHC': 'Chicago Cubs', 'CHW': 'Chicago White Sox', 'CWS': 'Chicago White Sox',
    'CIN': 'Cincinnati Reds', 'CLE': 'Cleveland Guardians', 'COL': 'Colorado Rockies',
    'DET': 'Detroit Tigers', 'HOU': 'Houston Astros', 'KCR': 'Kansas City Royals', 'KC': 'Kansas City Royals',
    'LAA': 'Los Angeles Angels', 'LAD': 'Los Angeles Dodgers', 'MIA': 'Miami Marlins',
    'MIL': 'Milwaukee Brewers', 'MIN': 'Minnesota Twins', 'NYM': 'New York Mets',
    'NYY': 'New York Yankees', 'ATH': 'Athletics', 'OAK': 'Athletics', 'PHI': 'Philadelphia Phillies', 
    'PIT': 'Pittsburgh Pirates', 'SDP': 'San Diego Padres', 'SD': 'San Diego Padres',
    'SEA': 'Seattle Mariners', 'SFG': 'San Francisco Giants', 'SF': 'San Francisco Giants', 'STL': 'St. Louis Cardinals',
    'TBR': 'Tampa Bay Rays', 'TB': 'Tampa Bay Rays', 'TEX': 'Texas Rangers', 'TOR': 'Toronto Blue Jays',
    'WSN': 'Washington Nationals', 'WAS': 'Washington Nationals', 'WSH': 'Washington Nationals'
}

# Translate the abbreviations into full names
HittersADP['Team'] = HittersADP['Team'].map(TeamNames).fillna('Free Agent')
PitchersADP['Team'] = PitchersADP['Team'].map(TeamNames).fillna('Free Agent')

# Display top elements of dataframe
print(HittersADP.head())
print()
print()
print(PitchersADP.head())

      Player Name                 Team     Position   ADP
0   Shohei Ohtani  Los Angeles Dodgers        SP,DH  1.17
1     Aaron Judge     New York Yankees  LF,CF,RF,DH  1.83
2  Bobby Witt Jr.   Kansas City Royals           SS  4.00
3       Juan Soto        New York Mets        LF,RF  3.50
4    Jose Ramirez  Cleveland Guardians        3B,DH  5.50


          Player Name                   Team Position    ADP
0        Tarik Skubal         Detroit Tigers       SP   6.00
1         Paul Skenes     Pittsburgh Pirates       SP   8.50
2     Garrett Crochet         Boston Red Sox       SP  11.33
3  Yoshinobu Yamamoto    Los Angeles Dodgers       SP  22.67
4  Cristopher Sanchez  Philadelphia Phillies       SP  26.33


#### Clean Position Column

In [5]:
# Split the string by comma or slash, keep the first position, and strip trailing spaces
HittersADP['Position'] = HittersADP['Position'].str.split(r'[,/]').str[0].str.strip()
PitchersADP['Position'] = PitchersADP['Position'].str.split(r'[,/]').str[0].str.strip()

# Convert specific outfield spots to 'OF'
HittersADP['Position'] = HittersADP['Position'].replace({'LF': 'OF', 'CF': 'OF', 'RF': 'OF', 'IF': 'SS'})

# Sort historical data by Year descending (e.g., 2025, 2024, 2023...)
HittersHistoricalSorted = HittersHistorical.sort_values(by='Year', ascending=False)
PithersHistoricalSorted = PithersHistorical.sort_values(by='Year', ascending=False)

# Create dictionaries 
HittersPositions = HittersHistoricalSorted.drop_duplicates(subset=['Player Name']).set_index('Player Name')['Position'].to_dict()
PitchersPositions = PithersHistoricalSorted.drop_duplicates(subset=['Player Name']).set_index('Player Name')['Position'].to_dict()

# Map positions and fill in the "nothing found" cases
HittersADP['Position'] = HittersADP['Player Name'].map(HittersPositions).fillna(HittersADP['Position'])
PitchersADP['Position'] = PitchersADP['Player Name'].map(PitchersPositions).fillna(PitchersADP['Position'])

# Display top elements of dataframe
print(HittersADP.head())
print()
print()
print(PitchersADP.head())

      Player Name                 Team Position   ADP
0   Shohei Ohtani  Los Angeles Dodgers       DH  1.17
1     Aaron Judge     New York Yankees       OF  1.83
2  Bobby Witt Jr.   Kansas City Royals       SS  4.00
3       Juan Soto        New York Mets       OF  3.50
4    Jose Ramirez  Cleveland Guardians       3B  5.50


          Player Name                   Team Position    ADP
0        Tarik Skubal         Detroit Tigers       SP   6.00
1         Paul Skenes     Pittsburgh Pirates       SP   8.50
2     Garrett Crochet         Boston Red Sox       SP  11.33
3  Yoshinobu Yamamoto    Los Angeles Dodgers       SP  22.67
4  Cristopher Sanchez  Philadelphia Phillies       SP  26.33


## Create CSV

In [6]:
HittersADP.to_csv("AverageDraftPositionDataClean/HittersADP2026Clean.csv", index=False)
PitchersADP.to_csv("AverageDraftPositionDataClean/PitchersADP2026Clean.csv", index=False)

print("\nSaved as 'HittersADP2026Clean.csv' and 'PitchersADP2026Clean.csv'.")


Saved as 'HittersADP2026Clean.csv' and 'PitchersADP2026Clean.csv'.
